# NB18: Pathway specificity (TP53, BRCA1/2, label-strategy ablations)

Verifies that the IHGAMP HRD score reflects HRD-pathway biology rather
than co-occurring confounders. Four sub-analyses:

1. **TP53 partial correlation** (Methods + Supp Table S14): partial r
   between IHGAMP HRD score and TP53 mutation status, controlling for
   scarHRD continuous score. Manuscript: r=0.112; AUC for TP53 prediction
   restricted to HRD-positive patients = 0.504.
2. **BRCA1/2 prediction (off-target)**: scarHRD-trained model evaluated
   for BRCA1/2 mutation status as outcome on test set. Manuscript:
   AUC = 0.683 (n=2,871 with mutation calls).
3. **BRCA1/2-trained comparator**: same pipeline, target switched to
   BRCA1/2 mutation status. Manuscript: BRCA-trained model evaluated on
   BRCA1/2 prediction yields AUC=0.486; scarHRD-trained model on
   BRCA-wildtype patients for HRD prediction yields AUC=0.772.
4. **5-label-strategy ablation** (Supp Table S15): retrain on each of
   five label definitions and report test AUROC. Manuscript: 0.765 at
   scarHRD>=33, 0.773 at scarHRD>=42.

**Required env**: `WORKSPACE`, `MC3_MAF_PATH`.
**Inputs**: NB06 clean embeddings, NB07 frozen scorer, NB04+NB05 labels.
**Outputs**: `results/pathway_specificity/{tp53.json, brca12_offtarget.json,
brca_trained.json, label_strategies.csv, report.json}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
MC3_MAF_PATH = Path(os.environ["MC3_MAF_PATH"]).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
RESULTS_DIR = WORKSPACE / "results" / "pathway_specificity"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR_PRIMARY = 33
HRD_THR_CLINICAL = 42
BOOT_N = 200
SILENT = {"Silent", "Intron", "3'UTR", "5'UTR", "3'Flank", "5'Flank", "IGR", "RNA"}

print(f"MC3_MAF_PATH: {MC3_MAF_PATH}")
print(f"SEED={SEED}, PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}")


In [ ]:
X = pd.read_parquet(EMB_DIR / "patient_means_clean.parquet").set_index("patient")
labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")
common = sorted(set(X.index) & set(labels.index))
X = X.loc[common]; labels = labels.loc[common].copy()
labels = labels.loc[labels["HRD"].notna()].copy()
X = X.loc[labels.index]
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR_PRIMARY).astype(int)

idx_tr = labels.index[labels["split"] == "train"]
idx_va = labels.index[labels["split"] == "val"]
idx_te = labels.index[labels["split"] == "test"]
print(f"patients with HRD: {len(labels):,}  splits {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")

frozen = joblib.load(MODELS_DIR / "frozen_model.joblib")
pipe_scarhrd = frozen["pipe"]
platt_scarhrd = frozen["platt"]

def predict_prob_scarhrd(X_arr):
    z = pipe_scarhrd.predict(X_arr).reshape(-1, 1)
    return platt_scarhrd.predict_proba(z)[:, 1]


In [ ]:
print("reading mc3 MAF (this may take a minute) ...")
maf = pd.read_csv(
    MC3_MAF_PATH, sep="\t", dtype=str, comment="#", low_memory=False,
    usecols=["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"]
)
maf["patient"] = maf["Tumor_Sample_Barcode"].astype(str).str.upper().str.slice(0, 12)
maf_ns = maf.loc[~maf["Variant_Classification"].isin(SILENT)].copy()
all_mc3_pts = set(maf["patient"].unique())

def gene_set(symbol):
    return set(maf_ns.loc[maf_ns["Hugo_Symbol"] == symbol, "patient"])

tp53_pts = gene_set("TP53")
brca1_pts = gene_set("BRCA1")
brca2_pts = gene_set("BRCA2")
brca12_pts = brca1_pts | brca2_pts
print(f"mc3 patients: {len(all_mc3_pts):,}  TP53: {len(tp53_pts):,}  "
      f"BRCA1: {len(brca1_pts):,}  BRCA2: {len(brca2_pts):,}  BRCA1/2: {len(brca12_pts):,}")

labels["TP53_mut"] = labels.index.isin(tp53_pts).astype(int)
labels["BRCA1_mut"] = labels.index.isin(brca1_pts).astype(int)
labels["BRCA2_mut"] = labels.index.isin(brca2_pts).astype(int)
labels["BRCA12_mut"] = labels.index.isin(brca12_pts).astype(int)
labels["in_mc3"] = labels.index.isin(all_mc3_pts)
n_with_mc3 = int(labels["in_mc3"].sum())
print(f"patients with WSI + scarHRD + mc3 mutation calls: {n_with_mc3:,}  (manuscript ref: 2,871)")


In [ ]:
# TP53 partial correlation, controlling for scarHRD
sub = labels.loc[labels["in_mc3"]].copy()
X_sub = X.loc[sub.index].values.astype(np.float32)
sub["IHGAMP_pred"] = predict_prob_scarhrd(X_sub)

x_var = sub["IHGAMP_pred"].astype(float).values
y_var = sub["TP53_mut"].astype(float).values
z_var = sub["HRD"].astype(float).values

def corr(a, b): return float(np.corrcoef(a, b)[0, 1])
r_xy = corr(x_var, y_var)
r_xz = corr(x_var, z_var)
r_yz = corr(y_var, z_var)
denom = np.sqrt(max(0.0, (1 - r_xz**2) * (1 - r_yz**2)))
partial_r = (r_xy - r_xz * r_yz) / denom if denom > 0 else float("nan")

hrd_pos = sub.index[sub["HRD_top20"] == 1]
y_pos_tp53 = sub.loc[hrd_pos, "TP53_mut"].astype(int).values
p_pos = sub.loc[hrd_pos, "IHGAMP_pred"].values
if y_pos_tp53.sum() == 0 or y_pos_tp53.sum() == len(y_pos_tp53):
    auc_within_hrdpos = float("nan")
else:
    auc_within_hrdpos = float(roc_auc_score(y_pos_tp53, p_pos))

tp53_res = {
    "n_patients_with_mc3_and_HRD": int(len(sub)),
    "n_TP53_mut": int(sub["TP53_mut"].sum()),
    "raw_pearson_IHGAMP_vs_TP53": r_xy,
    "raw_pearson_IHGAMP_vs_scarHRD": r_xz,
    "raw_pearson_TP53_vs_scarHRD": r_yz,
    "partial_r_IHGAMP_TP53_given_scarHRD": partial_r,
    "AUC_TP53_within_HRDpos": auc_within_hrdpos,
    "manuscript_refs": {"partial_r": 0.112, "AUC_TP53_within_HRDpos": 0.504},
}
print(f"raw r(IHGAMP, TP53)              : {r_xy:.3f}")
print(f"raw r(IHGAMP, scarHRD)           : {r_xz:.3f}")
print(f"raw r(TP53, scarHRD)             : {r_yz:.3f}")
print(f"partial r(IHGAMP, TP53 | scarHRD): {partial_r:.3f}   (manuscript ref: 0.112)")
print(f"AUC TP53 within HRD+ patients    : {auc_within_hrdpos:.3f}   (manuscript ref: 0.504)")
(RESULTS_DIR / "tp53.json").write_text(json.dumps(tp53_res, indent=2))


In [ ]:
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n: continue
        try: vals.append(fn(yb, pb))
        except Exception: pass
    if not vals: return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# scarHRD-trained -> BRCA1/2 prediction (off-target)
test_w_mut = sub.loc[sub["split"] == "test"].copy()
y_brca = test_w_mut["BRCA12_mut"].astype(int).values
p_test = test_w_mut["IHGAMP_pred"].values

if y_brca.sum() == 0 or y_brca.sum() == len(y_brca):
    brca_off = {"error": "single-class on test"}
else:
    auc = float(roc_auc_score(y_brca, p_test))
    ap = float(average_precision_score(y_brca, p_test))
    auc_lo, auc_hi = boot_ci(y_brca, p_test, roc_auc_score)
    brca_off = {
        "n": int(len(y_brca)), "n_BRCA12_mut": int(y_brca.sum()),
        "AUROC_BRCA12_offtarget": auc,
        "AUROC_lo": auc_lo, "AUROC_hi": auc_hi, "AP": ap,
        "manuscript_ref_AUROC": 0.683,
    }
    print(f"scarHRD-trained -> BRCA1/2 prediction (TEST)")
    print(f"  n={len(y_brca)} BRCA1/2-mut={int(y_brca.sum())}")
    print(f"  AUROC = {auc:.3f} ({auc_lo:.3f}-{auc_hi:.3f})  (manuscript ref: 0.683)")

(RESULTS_DIR / "brca12_offtarget.json").write_text(json.dumps(brca_off, indent=2))


In [ ]:
# BRCA1/2-trained comparator
def train_pipeline(X_tr_arr, y_tr_target, y_tr_binary):
    p = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("pca", PCA(n_components=PCA_N, random_state=SEED)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
    ]).fit(X_tr_arr, y_tr_target)
    z = p.predict(X_tr_arr).reshape(-1, 1)
    pl = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z, y_tr_binary)
    return p, pl

def score(p, pl, X_arr):
    return pl.predict_proba(p.predict(X_arr).reshape(-1, 1))[:, 1]

tr_idx_mc = sub.index[sub["split"] == "train"]
X_tr = X.loc[tr_idx_mc].values.astype(np.float32)
y_tr_brca = sub.loc[tr_idx_mc, "BRCA12_mut"].astype(int).values

if y_tr_brca.sum() < 5 or (len(y_tr_brca) - y_tr_brca.sum()) < 5:
    brca_trained = {"error": "too few BRCA1/2-mutated training patients"}
else:
    pipe_brca, platt_brca = train_pipeline(X_tr, y_tr_brca.astype(float), y_tr_brca)

    te_idx = sub.index[sub["split"] == "test"]
    X_te = X.loc[te_idx].values.astype(np.float32)
    y_te_brca = sub.loc[te_idx, "BRCA12_mut"].astype(int).values
    p_te_brca = score(pipe_brca, platt_brca, X_te)

    if y_te_brca.sum() == 0 or y_te_brca.sum() == len(y_te_brca):
        auc_brca_self = float("nan"); auc_lo = auc_hi = float("nan")
    else:
        auc_brca_self = float(roc_auc_score(y_te_brca, p_te_brca))
        auc_lo, auc_hi = boot_ci(y_te_brca, p_te_brca, roc_auc_score)

    bw_te = sub.loc[(sub["split"] == "test") & (sub["BRCA12_mut"] == 0)]
    y_bw = bw_te["HRD_top20"].astype(int).values
    p_bw = bw_te["IHGAMP_pred"].values
    if y_bw.sum() == 0 or y_bw.sum() == len(y_bw):
        auc_scarhrd_in_bw = float("nan"); bw_lo = bw_hi = float("nan")
    else:
        auc_scarhrd_in_bw = float(roc_auc_score(y_bw, p_bw))
        bw_lo, bw_hi = boot_ci(y_bw, p_bw, roc_auc_score)

    brca_trained = {
        "n_train": int(len(tr_idx_mc)),
        "n_train_BRCA12_pos": int(y_tr_brca.sum()),
        "BRCA_trained_self_AUROC_test": auc_brca_self,
        "BRCA_trained_self_CI": [auc_lo, auc_hi],
        "scarHRD_trained_AUROC_in_BRCA_wildtype": auc_scarhrd_in_bw,
        "scarHRD_trained_BW_CI": [bw_lo, bw_hi],
        "n_BRCA_wildtype_test": int(len(y_bw)),
        "manuscript_refs": {
            "BRCA_trained_self_AUROC": 0.486,
            "scarHRD_trained_AUROC_in_BRCA_wildtype": 0.772,
        },
    }
    print(f"BRCA-trained -> BRCA1/2 self-prediction (TEST)")
    print(f"  AUROC = {auc_brca_self:.3f} ({auc_lo:.3f}-{auc_hi:.3f})  (manuscript ref: 0.486)")
    print(f"scarHRD-trained -> HRD in BRCA-wildtype (TEST)")
    print(f"  AUROC = {auc_scarhrd_in_bw:.3f} ({bw_lo:.3f}-{bw_hi:.3f})  (manuscript ref: 0.772)")

(RESULTS_DIR / "brca_trained.json").write_text(json.dumps(brca_trained, indent=2))


In [ ]:
# 5-label-strategy ablation
def make_labels(strategy):
    hrd = labels["HRD"].astype(float)
    brca = labels["BRCA12_mut"].astype(int)
    if strategy == "scarHRD>=33":
        y_bin = (hrd >= 33).astype(int)
    elif strategy == "scarHRD>=42":
        y_bin = (hrd >= 42).astype(int)
    elif strategy == "scarHRD>=33 AND BRCA1/2":
        y_bin = ((hrd >= 33) & (brca == 1)).astype(int)
    elif strategy == "scarHRD>=33 OR BRCA1/2":
        y_bin = ((hrd >= 33) | (brca == 1)).astype(int)
    elif strategy == "scarHRD>=42 AND BRCA1/2":
        y_bin = ((hrd >= 42) & (brca == 1)).astype(int)
    else:
        raise ValueError(strategy)
    return y_bin.values

strategies = [
    "scarHRD>=33",
    "scarHRD>=42",
    "scarHRD>=33 AND BRCA1/2",
    "scarHRD>=33 OR BRCA1/2",
    "scarHRD>=42 AND BRCA1/2",
]

rows = []
for strat in strategies:
    bin_ = make_labels(strat)
    mask_train = (labels["split"] == "train") & labels["in_mc3"]
    mask_test = (labels["split"] == "test") & labels["in_mc3"]
    tr_ix = labels.index[mask_train]
    te_ix = labels.index[mask_test]

    y_tr_cont = labels.loc[tr_ix, "HRD"].astype(float).values
    y_tr_bin = pd.Series(bin_, index=labels.index).loc[tr_ix].values
    y_te_bin = pd.Series(bin_, index=labels.index).loc[te_ix].values

    if y_tr_bin.sum() < 10 or y_te_bin.sum() < 5:
        rows.append({"strategy": strat,
                     "n_train_pos": int(y_tr_bin.sum()),
                     "n_test_pos": int(y_te_bin.sum()),
                     "AUROC_test": float("nan"),
                     "reason": "too few positives"})
        continue

    X_tr_arr = X.loc[tr_ix].values.astype(np.float32)
    X_te_arr = X.loc[te_ix].values.astype(np.float32)

    p_, pl_ = train_pipeline(X_tr_arr, y_tr_cont, y_tr_bin)
    p_te = score(p_, pl_, X_te_arr)

    if y_te_bin.sum() == 0 or y_te_bin.sum() == len(y_te_bin):
        auc = float("nan")
    else:
        auc = float(roc_auc_score(y_te_bin, p_te))
    rows.append({
        "strategy": strat,
        "n_train": int(len(tr_ix)),
        "n_train_pos": int(y_tr_bin.sum()),
        "n_test": int(len(te_ix)),
        "n_test_pos": int(y_te_bin.sum()),
        "AUROC_test": auc,
    })
    print(f"  {strat:<28s} train_pos={int(y_tr_bin.sum()):4d} "
          f"test_pos={int(y_te_bin.sum()):4d}  AUROC={auc:.3f}")

label_df = pd.DataFrame(rows)
label_df.to_csv(RESULTS_DIR / "label_strategies.csv", index=False)
print()
print("manuscript refs: 0.765 at scarHRD>=33, 0.773 at scarHRD>=42 (Supp Table S15)")


In [ ]:
report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA,
               "HRD_THR_PRIMARY": HRD_THR_PRIMARY, "HRD_THR_CLINICAL": HRD_THR_CLINICAL,
               "BOOT_N": BOOT_N},
    "tp53": tp53_res,
    "brca12_offtarget": brca_off,
    "brca_trained": brca_trained,
    "label_strategies": rows,
    "manuscript_refs": {
        "partial_r_TP53_given_scarHRD": 0.112,
        "AUC_TP53_in_HRDpos": 0.504,
        "BRCA12_offtarget_AUC": 0.683,
        "BRCA_trained_self_AUC": 0.486,
        "scarHRD_in_BRCA_wildtype_AUC": 0.772,
        "label_strategy_AUC_at_33": 0.765,
        "label_strategy_AUC_at_42": 0.773,
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
